# 🧬 Arise 4-Encoder 1-Layer (RNA PCA) Model Runner & Visualizer

This notebook runs the **4-Encoder 1-Layer Arise Architecture** (RNA PCA + 1-Layer GCNs) and provides comprehensive visualizations:
- 📈 **Training Metrics Curve**: Loss, Epoch Silhouette Score (with best epoch highlight), and ARI.
- 🗺️ **Ground Truth vs Predicted Spatial Domains**: Side-by-side spatial coordinates comparison.
- 🎨 **UMAP Visualizations**: Integrated embedding space colored by ground truth and predicted clusters.
- 🎻 **Violin Plots**: Cluster-level distribution analysis of principal components and cluster silhouette profiles.

## 1. Environment & Dependencies Setup

In [ ]:
# Install dependencies
!pip install -q scanpy anndata scikit-misc gdown torch_geometric seaborn
print("Dependencies successfully installed!")

## 2. Imports & Configuration
Select your dataset, RNA PCA components (e.g. 60 or 100), seed, and training epochs.

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, silhouette_samples

# Ensure models directory is accessible
sys.path.append(os.path.abspath('models'))
sys.path.append(os.path.abspath('.'))

from AriseSpatialGlue_4Encoder_1Layer import (
    DATASET_REGISTRY,
    DATASET_LIST,
    preprocess_with_rna_pca,
    build_4encoder_graphs,
    Dual4Encoder1Layer,
    train_model,
    evaluate_model,
    plot_training_curves,
    set_seed
)

# ====================================================================
# ⚙️ CONFIGURATION
# ====================================================================
# Available datasets:
# 0: '10x_human_lymph_node_A1'
# 1: '10x_human_lymph_node_D1'
# 2: 'Mouse_Brain_E11_S1'
# 3: 'Mouse_Brain_E13_S1'
# 4: 'Mouse_Brain_E15_S1'
# 5: 'Mouse_Brain_E18_S1'
DATASET_INDEX = 0             # Choose index from 0 to 5
RNA_PCA_COMPS = 60            # RNA PCA components: 60 or 100
SEED = 42                     # Random seed (e.g. 42, 2024, 13)
EPOCHS = 350                  # Training epochs (Arise default: 350)
LEARNING_RATE = 1e-3          # Learning rate
BETA = 25.0                   # Reconstruction weight
GAMMA = 10.0                  # Spatial regularization weight
DELTA = 1.0                   # L1/L2 weight
HIDDEN_DIM = 512              # GCN hidden dimension
OUT_DIM = 64                  # Output embedding size
# ====================================================================

dataset_name = DATASET_LIST[DATASET_INDEX]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Selected Dataset : {dataset_name}")
print(f"RNA PCA Comps    : {RNA_PCA_COMPS}")
print(f"Seed: {SEED} | Epochs: {EPOCHS} | Device: {device}")

## 3. Dataset Download & Preprocessing (RNA PCA Reduction)

In [ ]:
cfg = DATASET_REGISTRY[dataset_name]
base = f"data/{dataset_name}"
os.makedirs(base, exist_ok=True)
rna_path = os.path.join(base, "adata_RNA.h5ad")
other_path = os.path.join(base, cfg["other_file"])
annotation_path = os.path.join(base, cfg["anno_file"])

# Download dataset files if needed
if not os.path.exists(rna_path) or not os.path.exists(other_path) or not os.path.exists(annotation_path):
    print(f"Downloading {dataset_name} files to {base}...")
    !gdown --folder "{cfg['url']}" --output "{base}"

# Load AnnData
adata_RNA = sc.read_h5ad(rna_path)
adata_omics2 = sc.read_h5ad(other_path)
adata_RNA.var_names_make_unique()
adata_omics2.var_names_make_unique()

anno_df = pd.read_csv(annotation_path, index_col=0)
true_labels = anno_df[cfg["gt_col"]].values
adata_RNA.obs['ground_truth'] = true_labels
adata_RNA.obs['ground_truth'] = adata_RNA.obs['ground_truth'].astype('category')
num_clusters = len(np.unique(true_labels))

# Preprocess RNA with PCA
RNA_expr, ADT_expr = preprocess_with_rna_pca(
    adata_RNA, adata_omics2, dataset_name, rna_pca_comps=RNA_PCA_COMPS
)
cell_positions = adata_RNA.obsm['spatial']

# Construct 4 Graphs
data = build_4encoder_graphs(RNA_expr, ADT_expr, cell_positions, device=device)
print(f"\nSuccessfully preprocessed {adata_RNA.n_obs} spots.")
print(f"• RNA Input Dim (PCA) : {data.x_RNA.shape[1]}")
print(f"• Aux Input Dim       : {data.x_ADT.shape[1]}")
print(f"• Target Clusters     : {num_clusters}")

## 4. Train 4-Encoder 1-Layer Model

In [ ]:
set_seed(SEED)

# Initialize Model (All 4 Encoders are 1-Layer GCNs)
model = Dual4Encoder1Layer(
    in_channels=data.x_RNA.shape[1],
    hidden_channels=HIDDEN_DIM,
    out_channels=OUT_DIM,
    q=data.x_ADT.shape[1],
    num_clusters=num_clusters,
    beta=BETA,
    gamma=GAMMA,
    delta=DELTA,
    dropout=0.0
).to(device)

# Train Model
results = train_model(
    model=model,
    data=data,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    num_clusters=num_clusters,
    true_labels=true_labels,
    verbose=True
)

# Store embeddings and predictions in AnnData
best_embeddings = results['best_embeddings']
best_labels = results['best_labels']

adata_RNA.obsm['Arise_1Layer'] = best_embeddings
adata_RNA.obs['predicted_domain'] = pd.Categorical(best_labels.astype(str))

# Compute Evaluation Metrics
ari = adjusted_rand_score(true_labels, best_labels)
nmi = normalized_mutual_info_score(true_labels, best_labels)
sil = results['best_sil']

print("=" * 60)
print(f"RESULTS SUMMARY ({dataset_name})")
print("=" * 60)
print(f"  Best Silhouette Score : {sil:.4f} (Epoch {results['best_epoch']})")
print(f"  ARI at Best Epoch     : {ari:.4f}")
print(f"  NMI at Best Epoch     : {nmi:.4f}")
print("=" * 60)

## 5. 📈 Plot Training Curves (Loss, Silhouette, ARI)

In [ ]:
# 📈 Plot Loss Curve, Silhouette Score Curve, and ARI Curve
plot_training_curves(
    results,
    dataset_name=f"{dataset_name} (RNA PCA {RNA_PCA_COMPS} Comps, Seed {SEED})",
    save_path=f"results/curves/{dataset_name}_seed{SEED}_training_curves.png",
    show=True
)


## 6. 🗺️ Ground Truth vs Predicted Spatial Domains Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# Ground Truth Spatial Map
sc.pl.spatial(
    adata_RNA,
    color='ground_truth',
    spot_size=1.5,
    ax=axes[0],
    show=False,
    title=f'Ground Truth ({dataset_name})'
)

# Predicted Spatial Domains Map
sc.pl.spatial(
    adata_RNA,
    color='predicted_domain',
    spot_size=1.5,
    ax=axes[1],
    show=False,
    title=f'Arise 4-Encoder 1-Layer Domains (ARI: {ari:.4f})'
)

plt.suptitle(f"Spatial Domains Comparison - {dataset_name}", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. 🎨 UMAP Embedding Visualizations

In [ ]:
# Compute UMAP on Arise 1-Layer joint embeddings
sc.pp.neighbors(adata_RNA, use_rep='Arise_1Layer')
sc.tl.umap(adata_RNA)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# UMAP colored by Ground Truth
sc.pl.umap(
    adata_RNA,
    color='ground_truth',
    ax=axes[0],
    show=False,
    title='UMAP: Ground Truth Annotation'
)

# UMAP colored by Predicted Domains
sc.pl.umap(
    adata_RNA,
    color='predicted_domain',
    ax=axes[1],
    show=False,
    title=f'UMAP: Predicted Domains (Silhouette: {sil:.4f})'
)

plt.suptitle(f"UMAP Joint Representation - {dataset_name}", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. 🎻 Violin Plots: Cluster Silhouette & Feature Distributions

In [ ]:
# 1. Compute sample-wise silhouette coefficients for each spot
sample_sil_values = silhouette_samples(best_embeddings, best_labels)
adata_RNA.obs['silhouette_coefficient'] = sample_sil_values

# 2. Store top embedding components in obs for distribution analysis
adata_RNA.obs['Latent_Dim_1'] = best_embeddings[:, 0]
adata_RNA.obs['Latent_Dim_2'] = best_embeddings[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Violin Plot 1: Silhouette Score Distribution per Predicted Domain
sns.violinplot(
    data=adata_RNA.obs,
    x='predicted_domain',
    y='silhouette_coefficient',
    palette='Set2',
    inner='quartile',
    ax=axes[0]
)
axes[0].axhline(sil, color='red', linestyle='--', label=f'Mean Sil: {sil:.4f}')
axes[0].set_title("Silhouette Coefficient per Predicted Domain", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Predicted Domain")
axes[0].set_ylabel("Silhouette Coefficient")
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle='--', alpha=0.3)

# Violin Plot 2: Latent Dimension 1 Expression per Domain
sns.violinplot(
    data=adata_RNA.obs,
    x='predicted_domain',
    y='Latent_Dim_1',
    palette='tab10',
    inner='box',
    ax=axes[1]
)
axes[1].set_title("Latent Dimension 1 Distribution per Domain", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Predicted Domain")
axes[1].set_ylabel("Latent Embedding Dim 1")
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.suptitle(f"Violin Plots: Cluster Profiles - {dataset_name}", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()